# NB1 — Data EDA + Panel Fingerprint (Abrigo Rev-2 migration)

**Purpose.** This notebook validates the Rev-5.3.2 panel construction; estimation lives in NB2 (`02_estimation.ipynb`); specification tests live in NB3 (`03_tests_and_sensitivity.ipynb`).

**Upstream input.** 14 Phase 5a panel parquets at `contracts/.scratch/2026-04-25-task110-rev2-data/` (`panel_row_01_primary.parquet` … `panel_row_14_wc_cpi_weights_sens.parquet`) plus the Phase 5a documentation (`manifest.md`, `data_dictionary.md`, `validation.md`, `_audit_summary.json`) and the Rev-5.3.2 published DuckDB state at `contracts/data/structural_econ.duckdb`.

**Downstream consumers.** NB2 consumes the panel-fingerprint JSON emitted by NB1 §1. NB3 consumes the Y₃ component decomposition emitted by NB1 §2 plus the outlier diagnostics emitted by NB1 §6.

**Pre-committed gate verdict.** FAIL (one-sided T3b). Reproduced byte-exact from `notebooks/abrigo_y3_x_d/estimates/gate_verdict.json` in NB2 / NB3; never re-estimated in this migration.

---

## §0 — Notebook header + panel-fingerprint validation

### Why-markdown (4-part citation block)

**Reference.**

- Rev-2 spec, autonomous track A, at `contracts/.scratch/2026-04-25-task110-rev2-spec-A-autonomous.md` (655 lines; §10.6 ζ-group roadmap; §11.A convex-payoff insufficiency caveat).
- Phase 5a Data Engineer outputs at `contracts/.scratch/2026-04-25-task110-rev2-data/` (14 panel parquets + `manifest.md` row-summary + `data_dictionary.md` + `validation.md` + `_audit_summary.json` machine-readable per-row audit).
- Phase 5b Analytics Reporter outputs at `contracts/.scratch/2026-04-25-task110-rev2-analysis/` (`estimates.md`, `spec_tests.md`, `sensitivity.md`, `summary.md`, `gate_verdict.json` with `gate_verdict = "FAIL"`).
- Rev-5.3.5 β-disposition memo at `contracts/.scratch/2026-04-26-mr-beta-1-1-halt-resolution-beta.md` (the disposition that reclassifies the X_d source address `0xc92e8fc2947e32f2b574cca9f2f12097a71d5606` as Minteo-fintech, OUT of Mento-native scope).
- Mento-native address-registry spec at `contracts/docs/superpowers/specs/2026-04-25-mento-native-address-registry.md` (canonical address-registry; locks `0x8A567e2aE79CA692Bd748aB832081C45de4041eA` as Mento-native `StableTokenCOP`).
- Y₃ inequality-differential design at `contracts/docs/superpowers/specs/2026-04-24-y3-inequality-differential-design.md` (4-country panel CO/BR/KE/EU; equal-weight aggregation; pre-registered WC-CPI weights 60/25/15).
- X_d carbon-basket design at `contracts/docs/superpowers/specs/2026-04-24-carbon-basket-xd-design.md` (carbon-basket user-volume series).
- Project memory: `project_y3_inequality_differential_design`, `project_abrigo_inequality_hedge_thesis`, `project_abrigo_convex_instruments_inequality`, `project_abrigo_mento_native_only` (β-corrigendum).

**Why used.** Section 0 establishes the panel-fingerprint contract that pins every downstream cell to the Rev-5.3.2 published estimates. The fingerprint is the load-bearing acceptance criterion of the migration: NB1 / NB2 / NB3 reproduce the published numbers BYTE-EXACT, or the migration fails. The Phase 5a `_audit_summary.json` is the upstream authority for row counts, methodology tags, and date windows; the gate-verdict SHA pin is the downstream authority for the published verdict. By fingerprinting both at the entry point of NB1, any silent drift (library skew, parquet rewrite, seed mismatch, methodology-tag rename) surfaces here rather than propagating into NB2 or NB3.

**Relevance to results.** The published primary-row estimate (β̂_X_d = -2.7987e-8, HAC(4) SE = 1.4234e-8, t-stat = -1.9662, n = 76, window `[2024-09-27, 2026-03-13]`, gate verdict FAIL) is byte-exact-immutable per the Rev-5.3.x anti-fishing invariants (`MDES_FORMULATION_HASH = 4940360dcd2987...cefa`; `decision_hash = 6a5f9d1b05c1...443c`; `N_MIN = 75`; `POWER_MIN = 0.80`; `MDES_SD = 0.40`). §0 verifies the panel inputs that produced those estimates are themselves byte-exact: 14 panels with the published row counts (76 / 76 / 65 / 56 / 76 / 76 / 45 / 47 / 0 / 0 / 76 / 76 / 76 / 76); `source_methodology = y3_v2_co_dane_br_bcb_eu_eurostat_ke_skip_3country_ke_unavailable` for the primary panel (Rev-5.3.2 default-flip post-Task 11.O Step-0, commit `202874565`); pre-registered FAIL rows at n = 65 (Row 3 LOCF-tail-excluded) and n = 56 (Row 4 IMF-only); deferred-empty Rows 9 and 10 with n = 0 per Phase 5a `manifest.md` §1.2 (spec §10 ε.2 / ε.3).

**Connection to product.** Abrigo sells convex (option-like) instruments that hedge macroeconomic shocks viewed through the inequality lens: the differential between rich-asset returns and working-class consumption returns. The Rev-2 mean-β estimation is first-stage / linear-hedge calibration only; per Rev-2 spec §11.A, mean-β is necessary-but-insufficient for convex-payoff product pricing. Convex-payoff fitness lives in Rev-3 ζ-group (quantile / GARCH-X / lower-tail / option-implied vol; deferred to Task 11.O.ζ-α per major-plan α-track scope). Crucially, Rev-2 measured X_d at the `0xc92e8fc2...` address; under the Rev-5.3.5 β-disposition that address is reclassified as Minteo-fintech (out of Mento-native scope), and the Mento-native COPm address `0x8A567e2a...` is the actual hedge-demand surface (deferred to Task 11.P.spec-β / Task 11.P.exec-β). Rev-2 therefore closes as scope-mismatch close-out on the Minteo-fintech X_d — not as a test of Mento-native hedge demand.

In [1]:
"""NB1 §0 — panel-fingerprint validation.

Loads the 14 Phase 5a panels + the audit summary from
contracts/.scratch/2026-04-25-task110-rev2-data/, asserts byte-exact reproduction
of the published row counts / methodology tags / date windows, and emits the
gate-verdict SHA pin. Functional-Python style: frozen dataclasses, free pure
functions, full typing, no inheritance.
"""
from __future__ import annotations

import hashlib
import importlib.util
import json
import sys
from dataclasses import dataclass
from pathlib import Path

import duckdb

# ---- Locate env.py and load by file path (notebooks/abrigo_y3_x_d/ is not on sys.path) ----

def _locate_abrigo_dir() -> Path:
    """Find the abrigo_y3_x_d/ directory that holds env.py."""
    cwd = Path.cwd().resolve()
    if (cwd / "env.py").is_file():
        return cwd
    for parent in (cwd, *cwd.parents):
        candidate = parent / "contracts" / "notebooks" / "abrigo_y3_x_d"
        if (candidate / "env.py").is_file():
            return candidate
        candidate2 = parent / "notebooks" / "abrigo_y3_x_d"
        if (candidate2 / "env.py").is_file():
            return candidate2
    raise RuntimeError(f"Could not locate abrigo_y3_x_d/env.py starting from cwd={cwd}")


_ABRIGO_DIR: Path = _locate_abrigo_dir()
_ENV_PATH: Path = _ABRIGO_DIR / "env.py"
_spec = importlib.util.spec_from_file_location("abrigo_env", _ENV_PATH)
env = importlib.util.module_from_spec(_spec)
sys.modules["abrigo_env"] = env
_spec.loader.exec_module(env)

# ---- Pre-committed Phase 5a expected fingerprint (binding upstream contract) ----

PRIMARY_METHODOLOGY: str = "y3_v2_co_dane_br_bcb_eu_eurostat_ke_skip_3country_ke_unavailable"
PRIMARY_WINDOW: tuple[str, str] = ("2024-09-27", "2026-03-13")
PRIMARY_N: int = 76

EXPECTED_N_OBS: dict[str, int] = {
    "row_01_primary": 76,
    "row_02_bootstrap_recon": 76,
    "row_03_locf_tail_excluded": 65,
    "row_04_imf_only_sensitivity": 56,
    "row_05_lag_sensitivity": 76,
    "row_06_parsimonious_controls": 76,
    "row_07_arb_only": 45,
    "row_08_per_currency_copm": 47,
    "row_09_y3_bond_diagnostic": 0,
    "row_10_population_weighted": 0,
    "row_11_student_t": 76,
    "row_12_hac12_bandwidth": 76,
    "row_13_first_differenced": 76,
    "row_14_wc_cpi_weights_sens": 76,
}

DEFERRED_ROWS: frozenset[str] = frozenset({"row_09_y3_bond_diagnostic", "row_10_population_weighted"})


@dataclass(frozen=True, slots=True)
class PanelFingerprint:
    """Per-row panel fingerprint emitted from §0 (sanity-check structure)."""
    row_label: str
    parquet_path: Path
    n_obs: int
    dt_min: str | None
    dt_max: str | None
    y3_methodology: str | None
    deferred: bool


@dataclass(frozen=True, slots=True)
class FingerprintReport:
    """Aggregated panel-fingerprint report for §0 sanity check."""
    panels: tuple[PanelFingerprint, ...]
    gate_verdict_sha256: str
    gate_verdict_path: Path


# ---- Pure functions (no inheritance, no globals mutated) ----

def _phase5a_data_dir() -> Path:
    return env._CONTRACTS_DIR / ".scratch" / "2026-04-25-task110-rev2-data"


def _read_audit_summary(data_dir: Path) -> dict[str, dict]:
    with (data_dir / "_audit_summary.json").open() as fh:
        return json.load(fh)


def _query_panel(conn: duckdb.DuckDBPyConnection, parquet_path: Path) -> tuple[int, str | None, str | None]:
    if not parquet_path.is_file():
        raise FileNotFoundError(f"Phase 5a parquet missing: {parquet_path}")
    sql = f"SELECT COUNT(*) AS n, MIN(week_start) AS dt_min, MAX(week_start) AS dt_max FROM '{parquet_path}'"
    row = conn.sql(sql).fetchone()
    n = int(row[0])
    dt_min = str(row[1]) if row[1] is not None else None
    dt_max = str(row[2]) if row[2] is not None else None
    return n, dt_min, dt_max


def _sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as fh:
        for chunk in iter(lambda: fh.read(65536), b""):
            h.update(chunk)
    return h.hexdigest()


def build_fingerprint_report(data_dir: Path, gate_verdict_path: Path) -> FingerprintReport:
    audit = _read_audit_summary(data_dir)
    conn = duckdb.connect()
    panels: list[PanelFingerprint] = []
    for row_label, expected_n in EXPECTED_N_OBS.items():
        parquet_path = data_dir / f"panel_{row_label}.parquet"
        n_obs, dt_min, dt_max = _query_panel(conn, parquet_path)
        is_deferred = row_label in DEFERRED_ROWS
        audit_row = audit[row_label]
        # Byte-exact assertions vs. _audit_summary.json + EXPECTED_N_OBS
        assert n_obs == expected_n, f"{row_label}: parquet n={n_obs} != EXPECTED_N_OBS={expected_n}"
        assert n_obs == int(audit_row["n_obs"]), f"{row_label}: parquet n={n_obs} != audit n_obs={audit_row['n_obs']}"
        if not is_deferred:
            assert dt_min == audit_row["dt_min"], f"{row_label}: dt_min mismatch {dt_min} vs {audit_row['dt_min']}"
            assert dt_max == audit_row["dt_max"], f"{row_label}: dt_max mismatch {dt_max} vs {audit_row['dt_max']}"
        panels.append(
            PanelFingerprint(
                row_label=row_label,
                parquet_path=parquet_path,
                n_obs=n_obs,
                dt_min=dt_min,
                dt_max=dt_max,
                y3_methodology=audit_row.get("y3_methodology"),
                deferred=is_deferred,
            )
        )
    conn.close()
    sha = _sha256_file(gate_verdict_path)
    return FingerprintReport(panels=tuple(panels), gate_verdict_sha256=sha, gate_verdict_path=gate_verdict_path)


# ---- Execute the fingerprint validation ----

_DATA_DIR = _phase5a_data_dir()
report = build_fingerprint_report(_DATA_DIR, env.GATE_VERDICT_PATH)

# Primary-panel anchor assertions (the load-bearing claims of §0)
_primary = next(p for p in report.panels if p.row_label == "row_01_primary")
assert _primary.n_obs == PRIMARY_N, f"Primary n_obs drift: got {_primary.n_obs}, expected {PRIMARY_N}"
assert _primary.y3_methodology == PRIMARY_METHODOLOGY, (
    f"Primary methodology drift: got {_primary.y3_methodology!r}, expected {PRIMARY_METHODOLOGY!r}"
)
assert (_primary.dt_min, _primary.dt_max) == PRIMARY_WINDOW, (
    f"Primary window drift: got ({_primary.dt_min}, {_primary.dt_max}), expected {PRIMARY_WINDOW}"
)

# Pre-registered FAIL anchors (Row 3 LOCF-tail-excluded; Row 4 IMF-only)
_row3 = next(p for p in report.panels if p.row_label == "row_03_locf_tail_excluded")
_row4 = next(p for p in report.panels if p.row_label == "row_04_imf_only_sensitivity")
assert _row3.n_obs == 65, f"Row 3 pre-registered FAIL anchor drift: n={_row3.n_obs}, expected 65"
assert _row4.n_obs == 56, f"Row 4 pre-registered FAIL anchor drift: n={_row4.n_obs}, expected 56"

# Deferred-empty anchors (Rows 9, 10 per Phase 5a manifest §1.2 / spec §10 ε.2/ε.3)
for _label in ("row_09_y3_bond_diagnostic", "row_10_population_weighted"):
    _row = next(p for p in report.panels if p.row_label == _label)
    assert _row.n_obs == 0, f"{_label} deferred-empty anchor drift: n={_row.n_obs}, expected 0"
    assert _row.deferred, f"{_label} not flagged deferred"

# Emit summary table inline (sanity check; downstream JSON emission is sub-task 2 / NB1 §1)
print(f"Phase 5a data dir: {_DATA_DIR}")
print(f"gate_verdict.json SHA-256: {report.gate_verdict_sha256}")
print(f"gate_verdict.json path: {report.gate_verdict_path}")
print(f"Primary methodology: {PRIMARY_METHODOLOGY}")
print(f"Primary window: [{PRIMARY_WINDOW[0]}, {PRIMARY_WINDOW[1]}]  n={PRIMARY_N}")
print()
print(f"{'row_label':<32} {'n_obs':>6} {'dt_min':<12} {'dt_max':<12} deferred")
for p in report.panels:
    print(
        f"{p.row_label:<32} {p.n_obs:>6} "
        f"{(p.dt_min or '-'):<12} {(p.dt_max or '-'):<12} "
        f"{p.deferred}"
    )

Phase 5a data dir: /home/jmsbpp/apps/ThetaSwap/thetaSwap-core-dev/.worktree/ranFromAngstrom/contracts/.scratch/2026-04-25-task110-rev2-data
gate_verdict.json SHA-256: 29f716ec835bb693f8985aeea9c97215aaf804931e916dbbf5afdf0cdf6e0259
gate_verdict.json path: /home/jmsbpp/apps/ThetaSwap/thetaSwap-core-dev/.worktree/ranFromAngstrom/contracts/notebooks/abrigo_y3_x_d/estimates/gate_verdict.json
Primary methodology: y3_v2_co_dane_br_bcb_eu_eurostat_ke_skip_3country_ke_unavailable
Primary window: [2024-09-27, 2026-03-13]  n=76

row_label                         n_obs dt_min       dt_max       deferred
row_01_primary                       76 2024-09-27   2026-03-13   False
row_02_bootstrap_recon               76 2024-09-27   2026-03-13   False
row_03_locf_tail_excluded            65 2024-09-27   2025-12-26   False
row_04_imf_only_sensitivity          56 2024-09-27   2025-10-24   False
row_05_lag_sensitivity               76 2024-09-27   2026-03-13   False
row_06_parsimonious_controls         76 

### Interpretation

**Panel-fingerprint validation establishes the byte-exact upstream contract for the migration.** All 14 Phase 5a panels load from `contracts/.scratch/2026-04-25-task110-rev2-data/` with row counts matching `_audit_summary.json` exactly: the primary panel at n = 76 over the window `[2024-09-27, 2026-03-13]` under `source_methodology = y3_v2_co_dane_br_bcb_eu_eurostat_ke_skip_3country_ke_unavailable` (Rev-5.3.2 default-flip post-Task 11.O Step-0); the two pre-registered FAIL anchors at n = 65 (Row 3 LOCF-tail-excluded) and n = 56 (Row 4 IMF-only); and the two deferred-empty rows at n = 0 (Row 9 Y₃-bond diagnostic, Row 10 population-weighted Y₃ — both deferred per spec §10 ε.2 / ε.3). The gate-verdict SHA-256 is emitted and the file is pinned at `notebooks/abrigo_y3_x_d/estimates/gate_verdict.json` so any future drift to the published verdict (gate_verdict = FAIL; β̂_X_d = -2.7987e-8; HAC(4) SE = 1.4234e-8; n = 76) would surface here rather than propagating silently into NB2 / NB3.

**Scope framing under Rev-5.3.5.** This notebook re-presents the Rev-2 panels, which used the Minteo-fintech X_d source at address `0xc92e8fc2947e32f2b574cca9f2f12097a71d5606`. Per the Rev-5.3.5 β-disposition memo at `contracts/.scratch/2026-04-26-mr-beta-1-1-halt-resolution-beta.md`, that address is now classified as Minteo-fintech (out of Mento-native scope per `project_abrigo_mento_native_only`); the canonical Mento-native `StableTokenCOP` address is `0x8A567e2aE79CA692Bd748aB832081C45de4041eA` per the address-registry spec at `contracts/docs/superpowers/specs/2026-04-25-mento-native-address-registry.md`. Rev-2 therefore closes as **Minteo-fintech scope-mismatch close-out** on the byte-exact-immutable estimates: the published mean-β numbers stay in the audit trail unchanged, but the interpretation is **scope-mismatch close-out** on the Minteo-fintech X_d. Concretely, Rev-2 closes scope-mismatch on the Minteo-fintech X_d at `0xc92e8fc2...`; the Mento-native COPm X_d hypothesis (against `0x8A567e2a...`) is the actual surface of interest and is deferred to Task 11.P.spec-β / Task 11.P.exec-β (the β-track Rev-3 sub-plan).

**Convex-payoff caveat preserved (Rev-2 spec §11.A).** Even setting aside the Minteo-fintech scope-mismatch close-out, mean-β identification is *necessary-but-insufficient* for convex-payoff product pricing. Whatever T3b yields on a future Mento-native panel is first-stage / linear-hedge calibration only; convex-payoff fitness (option / cap / floor instruments) requires the Rev-3 ζ-group extensions — quantile regression, GARCH-X, lower-tail dependence, option-implied-vol triangulation — which are deferred to Task 11.O.ζ-α per the major-plan α-track scope. NB3 §5 reproduces the spec §11.A caveat verbatim downstream; §0 here flags it because the panel-fingerprint contract does not by itself bound convex-payoff hedge fitness. Cross-references: `project_abrigo_convex_instruments_inequality`, `project_abrigo_inequality_hedge_thesis`, `project_y3_inequality_differential_design`, `project_abrigo_mento_native_only`.